# Flujo y transporte conservativo de la componente $U = SO_4^{2-} - Ca^{2+}$ con intercambio.

**Descripción**.

En esta notebook se plantea el problema de flujo de agua subterránea y transporte conservativo de la componente $U = c_1 - c_2$, donde $c_1= SO_4^{2-}$ y  $c_2 =  Ca^{2+}$ y se resuelve usando MODFLOW 6 (GWF y GWT) en combinación de las herramientas de **FloPy**. En este caso se hará uso de un intercambio entre los modelos de flujo y transporte, de tal manera que ambos modelos se comunican en tiempo de ejecución.

<a href="https://github.com/luiggix/RTWMA">RTWMA</a> © 2024-2026 by <a href="https://www.geofisica.unam.mx">IGF-UNAM</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc/4.0/">CC BY-NC 4.0</a><img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/by.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;"><img src="https://mirrors.creativecommons.org/presskit/icons/nc.svg" alt="" style="max-width: 1em;max-height:1em;margin-left: .2em;">

# Planteamiento del problema.

El dominio es una columna 1D de longitud $L = 30 \text{ (m)}$, en dirección $x$, representando un acuífero de espesor y ancho unitarios, véase figura 1. 

<figure>
  <img src="./figures/U_transporte.svg" width=600px alt="dominio de estudio y condiciones" />
  <figcaption>Figura 1. Dominio de estudio, condiciones iniciales y de frontera. El dominio es de tamaño $L = 30 \text{ (m)}$ y será dividido en $30$ celdas de $1\;\text{(m)}$ cada una de tal manera que $x_{_{12}}$ va del inicio del dominio al centro de la celda número $12$. </figcaption>
</figure>

Sobre este dominio ocurren dos procesos acoplados:

- Flujo de agua subterránea, gobernado por la carga hidráulica $h(x,t)$.
- Transporte de soluto, gobernado por la concentración $U(x,t)$, que se
  mueve por advección (arrastrado por el flujo de agua) y dispersión
  hidrodinámica.

El campo de flujo (carga $h$ y velocidad $v$) controla el transporte, pero el soluto no modifica la
densidad ni la viscosidad del agua.

El problema se va a resolver en un periodo de $40$ días, con un paso de tiempo constante de $1$ día ($40$ pasos de tiempo).

## Modelo de flujo de agua subterránea.

Para un acuífero 1D, confinado, con conductividad hidráulica $K_x$ y
coeficiente de almacenamiento específico $S_s$, la ecuación de flujo es:

$$
S_s \frac{\partial h}{\partial t} = \frac{\partial}{\partial x}
\left( K_x \frac{\partial h}{\partial x} \right) + q_s(x,t)
$$

donde $q_s(x,t)$ representa fuentes o sumideros volumétricos de agua por
unidad de volumen (p. ej. un pozo).

**Condiciones de frontera:**

$$
\left.\frac{\partial h}{\partial x}\right|_{(0,t)} = 0
\qquad\text{(frontera cerrada / flujo nulo en } x=0\text{)}
$$

$$
h(x_{_{L}}, t) = 1.0 \qquad\text{(carga fija en el extremo } x=x_{_{L}}\text{)}
$$

**Condición inicial:** 

$$
h(x,t_0) = 1.0 \qquad\text{(m)}
$$

## Modelo de transporte de soluto.

Para la especie $U$, con porosidad $\theta$, coeficiente de dispersión hidrodinámica $D_{xx}$ y descarga específica $q_x$ la ecuación de transporte se escribe como:

$$
\begin{eqnarray*}
- \frac{\partial (\theta U)}{\partial t}
- \frac{\partial (q_x U)}{\partial x}
+ \frac{\partial}{\partial x}\left(\theta D_{xx} \frac{\partial U}{\partial x}\right)
+ q_s U_s & = & 0
\end{eqnarray*}
$$

con $D_{xx} = \alpha_L |v_x| + D^{*}$ ($\alpha_L$: dispersividad longitudinal,
$D^{*}$: difusión molecular efectiva), $v_x = q_x / \theta $ ($v_x$: velocidad de
poro de flujo de agua) y $U_s$ es la concentración volumétrica de soluto en la fuente/sumidero.

**Condiciones de frontera:**

$$
U(x_{_{1}}, t) = c_1(x_{_{1}}, t) - c_2(x_{_{1}}, t) \qquad\text{(concentración fija en } x=x_{_{1}})
$$

$$
D_x \left.\frac{\partial U}{\partial x}\right|_{(L,t)} = 0
\qquad\text{(flujo dispersivo nulo en la salida } x=L )
$$

donde $c_1(x_{_{1}}, t) = 1.000329$ y $c_2(x_{_{1}}, t) = 3.294\times 10^{-5}$ $\; \forall \; t$.

**Condición inicial:**

$$
U(x,t_0) = c_1(x,t_0)-c_2(x,t_0) \quad \forall\, x \neq x_{_{12}}
$$
$$
U(x_{_{12}}, t_0) = c_1(x_{_{12}}, t_0)-c_2(x_{_{12}}, t_0)
$$

donde $c_1(x_{_{12}}, t_0) = 200.0$ y $c_2(x_{_{12}}, t_0) = 1.647\times 10^{-7}$.

# Solución numérica con MODFLOW 6.

La solución numérica se obtiene usando MODFLOW 6 y FloPy. En esta versión haremos una implementación con una sola simulación por lo que usaremos un intercambio de información entre el modelo de flujo y el de transporte. La ventaja es que el modelo de transporte no requiere leer archivos para obtener la información de flujo, si no que GWT la toma directamente de memoria inmediatamente que es calculada por GWF. La siguiente figura ilustra esta estructura.

<figure>
  <img src="./figures/mf6_flujo_transporte_exch.svg" width=450px alt="dominio de estudio y condiciones" />
  <figcaption>Figura 2. En una simulación de MODFLOW 6 de flujo y transporte, es posible agregar un intercambio para compartir información entre ambos modelos. Al final de la simulación se generan archivos con los resultados del flujo y del transporte. Véase (Langevin et al. 2023). </figcaption>
</figure>

La desventaja es que el flujo se resuelve en cada paso de tiempo.

## Definición de datos iniciales.

### Discretización espacial.

<figure>
  <img src="./figures/spatial_dis.svg" width=600px alt="dominio de estudio y condiciones" />
  <figcaption>Figura 4. El espacio se discretiza en 30 celdas. Se resaltan las celdas de la frontera y donde se tiene una condición inicial de concentración alta. </figcaption>
</figure>

|Parámetro | Valor | Unidades | Variable |
|---:|:---:|:---:|:---|
|Length of system (rows) | $30.0$ | m | |
|Number of layers | $1$ | | `dis["nlay"]`|
|Number of rows | $1$ | | `dis["nrow"]`|
|Number of columns | $30$ | | `dis["ncol"]`|
|Column width | $1.0$ | m | `dis["delr"]`|
|Row width | $1.0$ | m | `dis["delc"]`|
|Top of the model | $1.0$ | m | `dis["top"]`|
|Layer bottom elevation (cm) | $0$ | m | `dis["botm"]`|

### Discretización del tiempo.

|Parámetro | Valor| Unidades | Variable |
|---:|:---:|:---:|:---|
|Number of stress periods (NPER) | $1$ | | `tdis["nper"]` |
|Total time | $40$ | days | `tdis["perioddata"][0][0]` |
|Number of time steps (NSTP) | $40$ | | `tdis["perioddata"][0][1]` |
|Multiplier (TSMULT)| $1$ | | `tdis["perioddata"][0][2]` |

### Parámetros físicos.

Los parámetros necesarios para esta simulación son similares a los del caso de la especie $c_1$, solo cambian los valores de la concentración en los puntos clave, como se muestra en la siguiente tabla.

|Parámetro | Símbolo | Valor | Unidades | Variable |
|---:|:--:|:---:|:---:|:---|
|Initial head | $h(x, t_0)$| $1.0$ | m | `phys["initial_head"]` |
|Head boundary condition (type I) | $h({x_{_{L}}, t})$ | 1.0 | m | `phys["bc_head_t1"]` |
|Hydraulic conductivity | $K_x$ |$1.0$| m d$^{-1}$ | `phys["hydraulic_conductivity"]` |
|Specific discharge | $q_s$ |$0.2$| m d$^{-1}$ | `phys["specific_discharge"]` |
|Source concentration | $U_s$ | -- | unitless | `phys["source_concentration"][0]` |
|Well inyection | $q_s U_s$ | -- | m d$^{-1}$ | `phys["well"]`|
|Porosity | $\theta$ | $0.5$ | unitless |  `phys["porosity"]` |
|Initial concentration | $U(x, t_0)$ | -- | unitless | `phys["initial_concentration"]` |
|Initial concentration pulse | $U(x_{_{12}}, t_0)$ | -- | unitless | `phys["initial_concentration"][11]` |
|Concentration boundary condition (type I) | $U(x_{_{1}}, t)$ | -- | unitless | `phys["bc_conc_t1"][0]` |
|Longitudinal Dispersivity | $\alpha_L$ |$0.2$ | m  | `phys["longitudinal_dispersivity"]` |
|Dispersion coefficient | $D_{xx}$ | $1.0$ | m$^{2}$ d$^{-1}$| `phys["dispersion_coefficient"]` |

donde:

* $U_s = c_{s1} - c_{s2}$
* $U(x, t_0) = c_1(x, t_0)-c_2(x, t_0)$
* $U(x_{_{12}}, t_0) = c_1(x_{_{12}}, t_0)-c_2(x_{_{12}}, t_0)$
* $U(x_{_{1}}, t) = c_1(x_{_{1}}, t)-c_2(x_{_{1}}, t)$

### Rutas, nombres de archivos y otros.

Definimos un diccionario con información de la ruta al ejecutable de MODFLOW y con los nombres de los modelos de flujo y transporte así como los directorios del espacio de trabajo.

| Variable | Valor|
|---|---|
|`paths["sim_name"]` |`"flow_trans"`| 
|`paths["sim_ws"]` |`"output_Uexch"`| 
|`paths["flow_name"]` |`"flow"`| 
|`paths["tran_name"]` |`"transport"`| 

## Componentes de la simulación.

Para realizar una simulación de flujo y transporte con intercambio con MODFLOW 6, se requiere construir una estructura de componentes como  la que se muestra en la figura 2 de esta notebook.

Iniciamos generando la simulación y la discretización temporal mediante las clases `flopy.mf6.MFSimulation` y `flopy.mf6.ModflowTdis`. De esta manera construimos los objetos `o_sim` para la simulación y `o_tdis` para la discretización del tiempo. Estos objetos se comparten por ambos modelos. La implementación se encuentra en el archivo [`ex_U.py`](ex_U.py).

Posteriormente, se completa la estructura de la figura 2 agregando las componentes de cada modelo y de cada solución numérica. Esto se hace con los objetos `o_gwf` de la clase `flopy.mf6.ModflowGwf` y `o_ims_f` de la clase `flopy.mf6.ModflowIms` para el flujo, véase el archivo [`ex_gwf.py`](ex_gwf.py); y con los objetos `o_gwt` de la clase `flopy.mf6.ModflowGwt` y `o_ims_t` de la clase `flopy.mf6.ModflowIms` para el transporte, véase el archivo [`ex_gwt.py`](ex_gwt.py). Obsérvese que se usan dos objetos de la clase `flopy.mf6.ModflowIms` con atributos diferentes para resolver de manera independiente cada uno de los modelos. 

Una vez completada la estructura de la figura 2, se procede a agregar los paquetes a cada modelo.

## Construcción del modelo de flujo con GWF.

Para resolver el problema de flujo requerimos de los siguientes paquetes:

- `DIS`: Discretización espacial de tipo estructurada.
- `NPF`: Propiedades del flujo, particularmente la conductividad hidráulica $K_x$.
- `IC`: Definición de las condiciones iniciales de la carga hidráulica.
- `CHD`: Condición de frontera de carga hidráulica fija $h(x_{_{L}},t)=1.0$.
- `WEL`: Pozo de inyección de $q_s U_s$ en $x_{_{1}}$.
- `OC` : Definición de los archivos para almacenar los resultados de la simulación.
- No se define el paquete `STO` debido a que el flujo se resuelve en régimen
  permanente.
- El límite izquierdo ($x=0$) no recibe ninguna condición de frontera, por
  lo que MODFLOW 6 le asigna flujo nulo por defecto:
  $\partial h/\partial x(0,t)=0$ se cumple automáticamente.

Esto se implementa en el archivo [`ex_gwf.py`](ex_gwf.py).

## Construcción del modelo de transporte conservativo con GWT

Para resolver el problema de transporte se requiere de los siguientes paquetes:

- `DIS`: Discretización espacial de tipo estructurada. Se usa la misma que se usó para el flujo.
- `IC`: Condición inicial $U(x,t_0)$ que consiste en un valor constante en todo el dominio
  y el pulso puntual en $x_{_{12}}$.
- `ADV` : Advección con el esquema `TVD`.
- `DSP` : Dispersión, se define el coeficiente $\alpha_L$ (dispersión longitudinal) .
- `MST`: Se define la porosidad $\theta$.
- `CNC`: Condición de frontera de concentración fija $U(x_{_{1}},t)$.
- En $x_{_{L}}$ no se define `CNC`: MODFLOW 6 aplica de forma natural un
  gradiente dispersivo nulo en la celda de salida, es decir
  $D_x\,\partial U/\partial x(L,t)=0$ se cumple automáticamente.
- `FMI` : Le indica a la simulación de dónde leer las cargas (`GWFHEAD`) y los caudales
(`GWFBUDGET`) que ya fueron calculados por la simulación de flujo anterior. 
- `SSM`: vincula la concentración auxiliar del pozo `WEL-1` con el
  transporte (requerido por MODFLOW 6 cuando el modelo de flujo tiene
  paquetes de frontera).

Esto se implementa en el archivo [`ex_gwt.py`](ex_gwt.py).

## Intercambio.

Finalmente, se debe agregar la componente del intercambio a la simulación; en este punto ya deben estar definidos cada uno de los modelos que van a realizar el intercambio. En este caso estos modelos son generados en las funciones `build(...)` implementadas en los archivos [`ex_gwf.py`](ex_gwf.py) y [`ex_gwt.py`](ex_gwt.py).
Esta funciones regresan los objetos de los modelos:`o_gwf` para el flujo y `o_gwt` para el transporte.

Con estos objetos y el objeto de la simulación, `o_sim`, construimos el intercambio de la siguiente manera (véase el archivo [`ex_U.py`](ex_U.py)):

```python
o_gwfgwt = flopy.mf6.ModflowGwfgwt(
    o_sim, 
    exgtype="GWF6-GWT6", 
    exgmnamea=o_gwf.name, 
    exgmnameb=o_gwt.name,
    filename=f"{paths["sim_name"]}.gwfgwt",
)
```

## Escritura de archivos y ejecución.

En este caso, la escritura de los archivos y la ejecución se hace para los dos modelos dado que pertenecen a una sola simulación y esto se realiza a través del objeto `o_sim`. 

En la siguiente celda ejecutamos el código desde el archivo `ex_U.py`.

In [ ]:
%run ex_U.py

Para realizar el análisis y visualización de los resultados véase la notebook [06_flow_tran_gypsum_Uexch_vis.ipynb](06_flow_tran_gypsum_Uexch_vis.ipynb).

# Referencias.

* Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water. https://doi.org/10.1111/gwat.13351.
